In [5]:
"""
================================================================================
SURVEY DATA - COMPLETE ANALYSIS SCRIPT (v2)
================================================================================

CHANGES FROM v1:
  - Semester + CGPA REMOVED from modeling features
    (indirect proxies, not direct attrition drivers)
  - Upsolving habit ADDED (Q11 for active, Q33 practice habit for stopped)
  - Q38 "Could see quitting 6mo prior" ADDED for stopped branch
  - Q39 "Friend circle doing CP" ADDED (social support proxy)

KI KI KORBE EI SCRIPT:
  1. Data load + clean
  2. Quitting reasons chart
  3. Active vs Stopped descriptive comparison (charts)
  4. Statistical significance tests
  5. ML model train (LR, DT, RF, KNN, SVM, XGBoost, LightGBM, MLP)
  6. Cross-validation + test set results
  7. Feature importance chart
  8. EARLY WARNING SYSTEM -- active students-er risk score

HOW TO RUN:
  1. Google Colab-e jan
  2. Survey CSV upload korun (left panel > Files > Upload)
  3. Pura script ekta cell-e paste kore run korun (Shift+Enter)
  4. Shob output file download kore Claude-ke din

OUTPUT FILES:
  - survey_model_results.csv
  - survey_feature_importance.csv
  - survey_results_summary.txt
  - survey_active_risk_scores.csv
  - survey_cv_f1_chart.png
  - survey_feature_importance_chart.png
  - survey_active_vs_stopped_chart.png
  - survey_quitting_reasons_chart.png
  - survey_risk_distribution_chart.png
================================================================================
"""

# ============================================================
# INSTALL
# ============================================================
import subprocess
subprocess.run(["pip", "install", "xgboost", "lightgbm", "scipy", "--quiet"], check=True)

import pandas as pd
import numpy as np
import re, warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

from scipy import stats
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import xgboost as xgb
import lightgbm as lgb

RANDOM_STATE = 42
NAVY  = '#1B2A4A'
CORAL = '#E8604C'

# ============================================================
# STEP 1: LOAD + CLEAN
# ============================================================
print("=" * 60)
print("STEP 1: Loading and cleaning survey data")
print("=" * 60)

CSV_PATH = "Survey on Predicting the Competitive Programming Journey Form Responses.csv"
df = pd.read_csv(CSV_PATH)
print(f"Loaded: {len(df)} responses, {len(df.columns)} columns")

STATUS_COL = "6. Your current CP status? "
print(df[STATUS_COL].value_counts())

# ============================================================
# STEP 2: QUITTING REASONS CHART
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Quitting reasons chart")
print("=" * 60)

reasons_col = "31. Main reasons for quitting "
reasons = df[reasons_col].dropna().value_counts()
print(reasons)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [CORAL if i == 0 else NAVY for i in range(len(reasons))]
bars = ax.barh(reasons.index[::-1], reasons.values[::-1],
               color=colors[::-1], height=0.6, zorder=3)
for bar in bars:
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            str(int(bar.get_width())), va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Number of Respondents', fontsize=11)
ax.set_title('Self-Reported Reasons for Quitting CP\n(Previously Active Students, n=51)',
             fontsize=12, fontweight='bold')
ax.xaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
fig.savefig('survey_quitting_reasons_chart.png', bbox_inches='tight')
plt.close()
print("Saved: survey_quitting_reasons_chart.png")

# ============================================================
# STEP 3: BUILD ALIGNED FEATURE TABLE
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Building aligned feature table")
print("=" * 60)

modeling_df = df[df[STATUS_COL] != "Never started CP"].copy()
print(f"Modeling subset: {len(modeling_df)}")
print(modeling_df[STATUS_COL].value_counts())

def coalesce(row, col_a, col_s, default=np.nan):
    """Take active-branch value if present, else stopped-branch value."""
    val = row.get(col_a, np.nan)
    return val if pd.notna(val) else row.get(col_s, default)

# --- Core skill features ---
modeling_df["dsa_grade"]        = modeling_df["4. Grade in Data Structures and Algorithms (DSA) course "]
modeling_df["math_skill"]       = modeling_df["5. Mathematics skills rating "]
modeling_df["prior_exp"]        = modeling_df["3. Did you know programming before university? "]
modeling_df["ds_understanding"] = modeling_df.apply(
    lambda r: coalesce(r,
        "14. Understanding of standard data structures ",
        "37. Understanding of standard data structures when you quit "), axis=1)

# --- Support + behavior features ---
modeling_df["mentor_support"] = modeling_df.apply(
    lambda r: coalesce(r,
        "22. Have a mentor/senior guide? ",
        "35. Had mentor/senior support? "), axis=1)

modeling_df["long_break"] = modeling_df[
    "36. Have you taken 15+ consecutive days of break during your CP journey?\""
]

modeling_df["thought_quit"] = modeling_df.apply(
    lambda r: coalesce(r,
        "28. Ever seriously thought about quitting CP? ",
        "34.  Ever seriously thought about quitting CP?"), axis=1)

# --- Upsolving ---
# Active: Q11 direct upsolving question
# Stopped: Q33 "Practice habit before quitting" -- used as proxy
# (Became irregular / Stopped completely = poor upsolving habit)
modeling_df["upsolving_active"]  = modeling_df["11. Do you do upsolving after contests? "]
modeling_df["practice_stopped"]  = modeling_df["33.  Practice habit in the month before quitting "]

def align_upsolving(row):
    """Combine active upsolving + stopped practice habit into one feature."""
    if pd.notna(row["upsolving_active"]):
        return row["upsolving_active"]  # Always / Sometimes / Rarely / Never
    elif pd.notna(row["practice_stopped"]):
        # Map stopped practice habit to upsolving-like categories
        mapping = {
            "Regular":           "Sometimes",
            "Became irregular":  "Rarely",
            "Stopped completely":"Never",
        }
        return mapping.get(row["practice_stopped"], np.nan)
    return np.nan

modeling_df["upsolving_habit"] = modeling_df.apply(align_upsolving, axis=1)

# --- Friend circle (social support) ---
modeling_df["friend_circle_cp"] = modeling_df["39. How many people in your friend circle did CP? "]

# ============================================================
# FEATURE LISTS
# ============================================================
NUMERIC_FEATURES     = ["math_skill", "ds_understanding"]
CATEGORICAL_FEATURES = [
    "dsa_grade",
    "prior_exp",
    "mentor_support",
    "long_break",
    "thought_quit",
    "upsolving_habit",
    "friend_circle_cp",
]

# Target
y_model = (modeling_df[STATUS_COL] == "Previously did CP, but stopped").astype(int)
X_model  = modeling_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()

print(f"\nFeatures: {NUMERIC_FEATURES + CATEGORICAL_FEATURES}")
print(f"Class: Active={sum(y_model==0)}, Stopped={sum(y_model==1)}")
print(f"\nNull counts:")
print(X_model.isnull().sum())

# ============================================================
# STEP 4: ACTIVE vs STOPPED COMPARISON CHART
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Active vs Stopped comparison")
print("=" * 60)

active_df  = modeling_df[modeling_df[STATUS_COL] == "Currently active in CP"]
stopped_df = modeling_df[modeling_df[STATUS_COL] == "Previously did CP, but stopped"]

compare_numeric = {"Math Skill\n(1-5)": "math_skill", "DS Understanding\n(1-5)": "ds_understanding"}
active_means  = [active_df[col].mean()  for col in compare_numeric.values()]
stopped_means = [stopped_df[col].mean() for col in compare_numeric.values()]

x = np.arange(len(compare_numeric))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, active_means,  width, label='Active (n=22)',  color=NAVY,  zorder=3)
b2 = ax.bar(x + width/2, stopped_means, width, label='Stopped (n=51)', color=CORAL, zorder=3)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Mean Score (1-5)', fontsize=11)
ax.set_title('Active vs Stopped: Academic Skill Comparison\n(Survey, n=73)',
             fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(list(compare_numeric.keys()), fontsize=11)
ax.legend(fontsize=10)
ax.set_ylim(0, 6)
ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
fig.savefig('survey_active_vs_stopped_chart.png', bbox_inches='tight')
plt.close()
print("Saved: survey_active_vs_stopped_chart.png")

for label, col in compare_numeric.items():
    a = active_df[col].mean()
    s = stopped_df[col].mean()
    print(f"  {label.replace(chr(10),' '):20s}: Active={a:.2f}, Stopped={s:.2f}")

# ============================================================
# STEP 5: STATISTICAL SIGNIFICANCE TESTS
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Statistical significance tests")
print("=" * 60)

stat_results = {}

print("T-tests (numeric):")
for col in NUMERIC_FEATURES:
    a_vals = active_df[col].dropna()
    s_vals = stopped_df[col].dropna()
    if len(a_vals) > 2 and len(s_vals) > 2:
        t, p = stats.ttest_ind(a_vals, s_vals)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"  {col:25s}: t={t:.3f}, p={p:.4f} {sig}")
        stat_results[col] = {"t_stat": round(t,3), "p_value": round(p,4), "sig": sig}

print("\nChi-square tests (categorical):")
for col in CATEGORICAL_FEATURES:
    try:
        ct = pd.crosstab(modeling_df[STATUS_COL], modeling_df[col].fillna("Unknown"))
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"  {col:25s}: chi2={chi2:.3f}, p={p:.4f} {sig}")
        stat_results[col] = {"chi2": round(chi2,3), "p_value": round(p,4), "sig": sig}
    except Exception as e:
        print(f"  {col:25s}: Could not compute ({e})")

# ============================================================
# STEP 6: ML PIPELINE
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: ML Pipeline setup")
print("=" * 60)

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), NUMERIC_FEATURES),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), CATEGORICAL_FEATURES),
])

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.2, stratify=y_model, random_state=RANDOM_STATE
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

# ============================================================
# STEP 7: TRAIN ALL MODELS
# ============================================================
print("\n" + "=" * 60)
print("STEP 7: Training models")
print("=" * 60)

pos_weight = sum(y_model==0) / sum(y_model==1)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'),
    "Decision Tree":       DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced'),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
    "SVM":                 SVC(probability=True, random_state=RANDOM_STATE, class_weight='balanced'),
    "XGBoost":             xgb.XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                              scale_pos_weight=pos_weight,
                                              eval_metric='logloss', verbosity=0),
    "LightGBM":            lgb.LGBMClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                               class_weight='balanced', verbose=-1),
    "MLP":                 MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                                         random_state=RANDOM_STATE, early_stopping=True),
}

voting = VotingClassifier(
    estimators=[
        ("lr",  LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced')),
        ("rf",  RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced')),
        ("svm", SVC(probability=True, random_state=RANDOM_STATE, class_weight='balanced')),
        ("xgb", xgb.XGBClassifier(n_estimators=200, random_state=RANDOM_STATE,
                                    scale_pos_weight=pos_weight, eval_metric='logloss', verbosity=0)),
    ],
    voting="soft"
)
models["Soft-Voting Ensemble"] = voting

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results   = {}
test_results = {}
trained_pipes = {}

print("\n5-Fold CV F1:")
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")
    cv_results[name] = {"cv_f1_mean": round(scores.mean(), 3), "cv_f1_std": round(scores.std(), 3)}
    print(f"  {name:25s}: {scores.mean():.3f} (+/- {scores.std():.3f})")

print("\nTest Set:")
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    test_results[name] = {
        "test_accuracy":  round(accuracy_score(y_test, preds), 3),
        "test_f1":        round(f1_score(y_test, preds, zero_division=0), 3),
        "test_precision": round(precision_score(y_test, preds, zero_division=0), 3),
        "test_recall":    round(recall_score(y_test, preds, zero_division=0), 3),
    }
    trained_pipes[name] = pipe
    r = test_results[name]
    print(f"  {name:25s}: Acc={r['test_accuracy']:.3f} F1={r['test_f1']:.3f} "
          f"Prec={r['test_precision']:.3f} Rec={r['test_recall']:.3f}")

# ============================================================
# STEP 8: FEATURE IMPORTANCE
# ============================================================
print("\n" + "=" * 60)
print("STEP 8: Feature Importance (Random Forest)")
print("=" * 60)

rf_pipe = trained_pipes["Random Forest"]
cat_names = list(rf_pipe.named_steps["prep"]
                 .named_transformers_["cat"]
                 .named_steps["onehot"]
                 .get_feature_names_out(CATEGORICAL_FEATURES))
all_names = NUMERIC_FEATURES + cat_names
importances = rf_pipe.named_steps["clf"].feature_importances_

agg = {}
for fname, imp in zip(all_names, importances):
    parent = fname
    for c in CATEGORICAL_FEATURES:
        if fname.startswith(c + "_"):
            parent = c
            break
    agg[parent] = agg.get(parent, 0) + imp

feat_imp = pd.DataFrame(list(agg.items()), columns=["feature","importance"]).sort_values("importance", ascending=False)
print(feat_imp.to_string(index=False))
feat_imp.to_csv("survey_feature_importance.csv", index=False)

# ============================================================
# STEP 9: EARLY WARNING SYSTEM
# ============================================================
print("\n" + "=" * 60)
print("STEP 9: Early Warning System")
print("=" * 60)

best_model_name = max(cv_results, key=lambda m: cv_results[m]['cv_f1_mean'])
print(f"Best model: {best_model_name} (CV F1={cv_results[best_model_name]['cv_f1_mean']:.3f})")

# Retrain on full data
best_pipe = Pipeline([("prep", preprocessor), ("clf", models[best_model_name])])
best_pipe.fit(X_model, y_model)

# Active students only
active_only = df[df[STATUS_COL] == "Currently active in CP"].copy()
active_only["math_skill"]       = active_only["5. Mathematics skills rating "]
active_only["ds_understanding"] = active_only["14. Understanding of standard data structures "]
active_only["dsa_grade"]        = active_only["4. Grade in Data Structures and Algorithms (DSA) course "]
active_only["prior_exp"]        = active_only["3. Did you know programming before university? "]
active_only["mentor_support"]   = active_only["22. Have a mentor/senior guide? "]
active_only["long_break"]       = active_only.get("27. Taken 15+ consecutive days break in last 3 months? ", np.nan)
active_only["thought_quit"]     = active_only["28. Ever seriously thought about quitting CP? "]
active_only["upsolving_habit"]  = active_only["11. Do you do upsolving after contests? "]
active_only["friend_circle_cp"] = active_only["39. How many people in your friend circle did CP? "]

X_active = active_only[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
risk_probs = best_pipe.predict_proba(X_active)[:, 1]

risk_labels = pd.cut(risk_probs,
                     bins=[0, 0.33, 0.66, 1.0],
                     labels=["Low Risk", "Medium Risk", "High Risk"])

risk_df = pd.DataFrame({
    "student_id":            range(1, len(active_only)+1),
    "dsa_grade":             active_only["4. Grade in Data Structures and Algorithms (DSA) course "].values,
    "math_skill":            active_only["5. Mathematics skills rating "].values,
    "ds_understanding":      active_only["14. Understanding of standard data structures "].values,
    "mentor":                active_only["22. Have a mentor/senior guide? "].values,
    "upsolving":             active_only["11. Do you do upsolving after contests? "].values,
    "thought_quit":          active_only["28. Ever seriously thought about quitting CP? "].values,
    "attrition_risk_pct":    np.round(risk_probs * 100, 1),
    "risk_category":         risk_labels,
}).sort_values("attrition_risk_pct", ascending=False)

risk_df.to_csv("survey_active_risk_scores.csv", index=False)

print(f"\nRisk distribution (n={len(active_only)} active students):")
print(risk_df['risk_category'].value_counts())
print("\nTop 5 highest-risk active students:")
print(risk_df.head(5).to_string(index=False))

# ============================================================
# STEP 10: CHARTS
# ============================================================
print("\n" + "=" * 60)
print("STEP 10: Generating charts")
print("=" * 60)

# CV F1 Chart
model_names = list(cv_results.keys())
cv_means = [cv_results[m]['cv_f1_mean'] for m in model_names]
cv_stds  = [cv_results[m]['cv_f1_std']  for m in model_names]

fig, ax = plt.subplots(figsize=(11, 5))
best_cv = max(cv_means)
colors = [CORAL if v == best_cv else NAVY for v in cv_means]
ax.bar(model_names, cv_means, color=colors, width=0.6, zorder=3)
ax.errorbar(range(len(model_names)), cv_means, yerr=cv_stds,
            fmt='none', color='black', capsize=4, linewidth=1.2, zorder=4)
for i, (name, val) in enumerate(zip(model_names, cv_means)):
    ax.text(i, val + 0.015, f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Mean F1-Score (5-Fold CV)', fontsize=12)
ax.set_title('Cross-Validation Performance Across Classifiers\n(Survey Dataset, n=73)', fontsize=13, fontweight='bold')
ax.set_xticks(range(len(model_names)))
ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
ax.yaxis.grid(True, linestyle='--', alpha=0.5, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
fig.savefig('survey_cv_f1_chart.png', bbox_inches='tight')
plt.close()
print("Saved: survey_cv_f1_chart.png")

# Feature Importance Chart
top_feats = feat_imp.head(8)
fig, ax = plt.subplots(figsize=(8, 5))
colors = [CORAL if i == 0 else NAVY for i in range(len(top_feats))]
ax.barh(top_feats['feature'][::-1], top_feats['importance'][::-1], color=colors[::-1], height=0.6, zorder=3)
for i, (val, feat) in enumerate(zip(top_feats['importance'][::-1], top_feats['feature'][::-1])):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)
ax.set_xlabel('Feature Importance', fontsize=11)
ax.set_title('Random Forest Feature Importance\n(Survey Dataset)', fontsize=12, fontweight='bold')
ax.xaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
fig.savefig('survey_feature_importance_chart.png', bbox_inches='tight')
plt.close()
print("Saved: survey_feature_importance_chart.png")

# Risk Distribution Chart
risk_counts = risk_df['risk_category'].value_counts()
risk_order  = ['High Risk', 'Medium Risk', 'Low Risk']
risk_vals   = [risk_counts.get(r, 0) for r in risk_order]
risk_colors = [CORAL, '#F5A623', NAVY]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(risk_order, risk_vals, color=risk_colors, width=0.5, zorder=3)
for bar, val in zip(bars, risk_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(val)), ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Active Students', fontsize=11)
ax.set_title(f'Attrition Risk Among Currently Active Students\n(n={len(active_only)}, Model: {best_model_name})',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, max(risk_vals) + 3)
ax.yaxis.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
fig.tight_layout()
fig.savefig('survey_risk_distribution_chart.png', bbox_inches='tight')
plt.close()
print("Saved: survey_risk_distribution_chart.png")

# ============================================================
# STEP 11: SAVE ALL RESULTS
# ============================================================
print("\n" + "=" * 60)
print("STEP 11: Saving results")
print("=" * 60)

rows = []
for name in models:
    row = {"model": name}
    row.update(cv_results[name])
    row.update(test_results[name])
    rows.append(row)
pd.DataFrame(rows).to_csv("survey_model_results.csv", index=False)

with open("survey_results_summary.txt", "w") as f:
    f.write("=" * 60 + "\n")
    f.write("SURVEY DATA - COMPLETE ANALYSIS SUMMARY (v2)\n")
    f.write("=" * 60 + "\n\n")
    f.write("FEATURES USED (v2 - updated):\n")
    f.write("  Numeric: math_skill, ds_understanding\n")
    f.write("  Categorical: dsa_grade, prior_exp, mentor_support,\n")
    f.write("               long_break, thought_quit, upsolving_habit,\n")
    f.write("               friend_circle_cp\n")
    f.write("  REMOVED: semester, cgpa (indirect proxies)\n\n")
    f.write("QUITTING REASONS (n=51):\n")
    for r, c in reasons.items():
        f.write(f"  {r}: {c}\n")
    f.write("\nSTATISTICAL TESTS:\n")
    for feat, res in stat_results.items():
        if 't_stat' in res:
            f.write(f"  {feat}: t={res['t_stat']}, p={res['p_value']} {res['sig']}\n")
        else:
            f.write(f"  {feat}: chi2={res['chi2']}, p={res['p_value']} {res['sig']}\n")
    f.write("\nCROSS-VALIDATION (5-Fold F1):\n")
    for name, r in cv_results.items():
        f.write(f"  {name:25s}: {r['cv_f1_mean']:.3f} (+/- {r['cv_f1_std']:.3f})\n")
    f.write("\nTEST SET:\n")
    for name, r in test_results.items():
        f.write(f"  {name:25s}: Acc={r['test_accuracy']:.3f} F1={r['test_f1']:.3f} "
                f"Prec={r['test_precision']:.3f} Rec={r['test_recall']:.3f}\n")
    f.write(f"\nBEST MODEL: {best_model_name} (CV F1={cv_results[best_model_name]['cv_f1_mean']:.3f})\n")
    f.write("\nTOP FEATURES:\n")
    for _, row in feat_imp.head(5).iterrows():
        f.write(f"  {row['feature']:30s}: {row['importance']:.4f}\n")
    f.write("\nEARLY WARNING RISK DISTRIBUTION:\n")
    for cat in risk_order:
        f.write(f"  {cat}: {risk_counts.get(cat, 0)} students\n")

print("Saved: survey_results_summary.txt")

print("\n" + "=" * 60)
print("ALL DONE! Download these files:")
print("=" * 60)
for fname in [
    "survey_model_results.csv",
    "survey_feature_importance.csv",
    "survey_results_summary.txt",
    "survey_active_risk_scores.csv",
    "survey_cv_f1_chart.png",
    "survey_feature_importance_chart.png",
    "survey_active_vs_stopped_chart.png",
    "survey_quitting_reasons_chart.png",
    "survey_risk_distribution_chart.png",
]:
    print(f"  files.download('{fname}')")

STEP 1: Loading and cleaning survey data
Loaded: 96 responses, 50 columns
6. Your current CP status? 
Previously did CP, but stopped    51
Never started CP                  23
Currently active in CP            22
Name: count, dtype: int64

STEP 2: Quitting reasons chart
31. Main reasons for quitting 
Rating not increasing/no progress        12
Others                                    9
Interest in other tracks (Web Dev/ML)     9
Lack of guidance or mentor                8
Academic pressure (CGPA focus)            7
Mental stress or burnout                  6
Name: count, dtype: int64
Saved: survey_quitting_reasons_chart.png

STEP 3: Building aligned feature table
Modeling subset: 73
6. Your current CP status? 
Previously did CP, but stopped    51
Currently active in CP            22
Name: count, dtype: int64

Features: ['math_skill', 'ds_understanding', 'dsa_grade', 'prior_exp', 'mentor_support', 'long_break', 'thought_quit', 'upsolving_habit', 'friend_circle_cp']
Class: Active=22, St